# 🌳 Árbol de expresión modular: construcción, recorridos y graficación

**Ejemplo base:** `sqrt(2*(x^2)+5*x-3)`, es decir, $\sqrt{2x^2 + 5x - 3}$

El proyecto está organizado como un **paquete de Python** llamado `arbol_expresion`, con un módulo por responsabilidad:

| Módulo | Responsabilidad |
|---|---|
| `config.py` | Funciones, constantes, operadores (precedencia y asociatividad) y colores |
| `errores.py` | Excepción `ErrorExpresion` y mensajes que señalan la posición del error |
| `lexico.py` | Normalización y tokenización (análisis léxico) |
| `sintaxis.py` | Menos unario, multiplicación implícita y validación sintáctica |
| `postfija.py` | Algoritmo Shunting-Yard: infija → postfija |
| `arbol.py` | Clase `Nodo`, construcción del árbol, recorridos y estadísticas |
| `evaluador.py` | Evaluación numérica con validación de dominio |
| `graficos.py` | Dibujo del árbol y gráfica de la función |
| `analizador.py` | Clase `ArbolExpresion` y función `analizar()` (une todo) |
| `interactivo.py` | Modo interactivo con `input()` |
| `pruebas.py` | Pruebas automáticas |
| `__init__.py` | Expone la interfaz pública del paquete |

**Flujo de datos:**

`texto → lexico → tokens → sintaxis → tokens válidos → postfija → árbol → evaluador / graficos`

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todas` (o `Ctrl + F9`).
Las celdas `%%writefile` crean cada archivo del paquete; después se importa y se usa.

## Paso 1 · Crear la carpeta del paquete

In [ ]:
import os
os.makedirs('arbol_expresion', exist_ok=True)
print("📁 Carpeta 'arbol_expresion' lista.")

## Paso 2 · Módulo `config.py`

Cada operador tiene **precedencia**, **asociatividad** y **aridad** (número de operandos):

| Operador | Significado | Precedencia | Asociatividad | Aridad |
|---|---|---|---|---|
| `+` `-` | suma, resta | 1 | izquierda | 2 |
| `*` `/` | multiplicación, división | 2 | izquierda | 2 |
| `neg` | negativo unario (`-x`) | 3 | derecha | 1 |
| `^` | potencia | 4 | derecha | 2 |

> `2^3^2 = 2^(3^2) = 512` (asociatividad derecha) y `-x^2 = -(x^2)` (el negativo tiene menor precedencia que `^`).
>
> **Extensible:** para agregar una función nueva basta con añadirla al diccionario `FUNCIONES`.

In [ ]:
%%writefile arbol_expresion/config.py
"""
config.py — Configuración del lenguaje de expresiones.

Para agregar una función nueva (por ejemplo, 'asin') basta con añadirla
al diccionario FUNCIONES; el resto de los módulos la reconocen solos.
"""
import math

# Funciones de un argumento: nombre -> implementación
FUNCIONES = {
    'sqrt': math.sqrt,   # raíz cuadrada
    'sin':  math.sin,    # seno (radianes)
    'cos':  math.cos,    # coseno (radianes)
    'tan':  math.tan,    # tangente (radianes)
    'log':  math.log10,  # logaritmo base 10
    'ln':   math.log,    # logaritmo natural
    'exp':  math.exp,    # e^x
    'abs':  abs,         # valor absoluto
}

# Constantes: nombre -> valor
CONSTANTES = {'pi': math.pi, 'e': math.e}

# Operadores: símbolo -> (precedencia, asociatividad, aridad)
OPERADORES = {
    '+':   (1, 'izq', 2),
    '-':   (1, 'izq', 2),
    '*':   (2, 'izq', 2),
    '/':   (2, 'izq', 2),
    'neg': (3, 'der', 1),   # menos unario: -x
    '^':   (4, 'der', 2),
}

# Tipos de token que son operandos (hojas del árbol)
TIPOS_OPERANDO = ('NUM', 'VAR', 'CONST')

# Colores del dibujo: tipo de nodo -> (color, etiqueta de la leyenda)
COLORES = {
    'OP':    ('#4C72B0', 'Operador binario'),
    'OP_UN': ('#8172B3', 'Operador unario (neg)'),
    'FUNC':  ('#DD8452', 'Función'),
    'NUM':   ('#55A868', 'Número'),
    'VAR':   ('#C44E52', 'Variable'),
    'CONST': ('#2A9D8F', 'Constante'),
}

## Paso 3 · Módulo `errores.py`

Define la excepción propia del proyecto y la función que dibuja una flecha `^` bajo la posición exacta del error.

In [ ]:
%%writefile arbol_expresion/errores.py
"""
errores.py — Excepción propia y utilidades para mensajes de error.
"""


class ErrorExpresion(Exception):
    """Error de sintaxis o de evaluación en una expresión."""
    pass


def senalar(expr, pos, mensaje):
    """Arma un mensaje de error con una flecha (^) bajo la posición del problema."""
    return f"{mensaje}\n      {expr}\n      {' ' * pos}^"


def simbolo(valor):
    """Representación legible de un operador ('neg' se muestra como '-')."""
    return '-' if valor == 'neg' else valor

## Paso 4 · Módulo `lexico.py` (análisis léxico)

**Normalizar** unifica la escritura: `**` → `^`, `√` → `sqrt`, `×` → `*`, `[ ]` y `{ }` → `( )`, coma decimal → punto, mayúsculas → minúsculas.

**Tokenizar** separa la cadena en números, variables, constantes, funciones, operadores y paréntesis.

Validaciones: expresión vacía, signo `=`, caracteres no permitidos, números mal formados e identificadores desconocidos.

In [ ]:
%%writefile arbol_expresion/lexico.py
"""
lexico.py — Análisis léxico: normalización y tokenización.

Validaciones de esta etapa:
  - texto vacío o que no es str
  - uso del signo '='
  - caracteres no permitidos
  - números mal formados (3.4.5, '.')
  - identificadores desconocidos (raiz, sen, xy...)
"""
import re
from dataclasses import dataclass

from .config import FUNCIONES, CONSTANTES
from .errores import ErrorExpresion, senalar


@dataclass
class Token:
    tipo: str    # NUM, VAR, CONST, FUNC, OP, PAR_IZQ, PAR_DER
    valor: str
    pos: int     # posición en la cadena, para señalar errores


# Reemplazos para aceptar distintas formas de escribir lo mismo
_REEMPLAZOS = [
    ('**', '^'), ('×', '*'), ('·', '*'), ('÷', '/'), ('−', '-'),
    ('√', ' sqrt '), ('[', '('), (']', ')'), ('{', '('), ('}', ')'),
    (',', '.'),   # coma decimal -> punto decimal
]


def normalizar(expr):
    """Unifica la escritura de la expresión y valida que no esté vacía."""
    if not isinstance(expr, str):
        raise ErrorExpresion("La expresión debe ser un texto (str).")
    expr = expr.strip()
    if not expr:
        raise ErrorExpresion("La expresión está vacía.")
    if '=' in expr:
        raise ErrorExpresion("Escribe solo una expresión, sin el signo '=' "
                             "(por ejemplo: 2*x^2+5*x-3 en lugar de y=2*x^2+5*x-3).")
    for viejo, nuevo in _REEMPLAZOS:
        expr = expr.replace(viejo, nuevo)
    return re.sub(r'\s+', ' ', expr).strip().lower()


def _leer_numero(expr, i):
    texto = re.match(r'\d*\.?\d*', expr[i:]).group()
    fin = i + len(texto)
    if texto == '.':
        raise ErrorExpresion(senalar(expr, i, "Punto decimal sin dígitos."))
    if fin < len(expr) and expr[fin] == '.':
        raise ErrorExpresion(senalar(expr, fin, "Número mal formado (más de un punto decimal)."))
    return Token('NUM', texto, i), fin


def _leer_identificador(expr, i):
    palabra = re.match(r'[a-z]+', expr[i:]).group()
    if palabra in FUNCIONES:
        tipo = 'FUNC'
    elif palabra in CONSTANTES:
        tipo = 'CONST'
    elif len(palabra) == 1:
        tipo = 'VAR'
    else:
        raise ErrorExpresion(senalar(
            expr, i,
            f"Identificador desconocido '{palabra}'.\n"
            f"      Funciones válidas: {', '.join(FUNCIONES)}. Constantes: {', '.join(CONSTANTES)}.\n"
            f"      Las variables son de una letra; para multiplicar variables escribe x*y."))
    return Token(tipo, palabra, i), i + len(palabra)


_SIMBOLOS = {'+': 'OP', '-': 'OP', '*': 'OP', '/': 'OP', '^': 'OP',
             '(': 'PAR_IZQ', ')': 'PAR_DER'}


def tokenizar(expr):
    """Convierte la cadena normalizada en una lista de Token."""
    tokens, i = [], 0
    while i < len(expr):
        c = expr[i]
        if c.isspace():
            i += 1
        elif c.isdigit() or c == '.':
            tok, i = _leer_numero(expr, i)
            tokens.append(tok)
        elif c.isascii() and c.isalpha():
            tok, i = _leer_identificador(expr, i)
            tokens.append(tok)
        elif c in _SIMBOLOS:
            tokens.append(Token(_SIMBOLOS[c], c, i))
            i += 1
        else:
            raise ErrorExpresion(senalar(expr, i, f"Carácter no válido '{c}'."))
    return tokens

## Paso 5 · Módulo `sintaxis.py` (preprocesamiento y validación)

- Convierte el `-` unario en el operador `neg` (el `+` unario se descarta).
- Inserta **multiplicaciones implícitas**: `2x` → `2*x`, `3(x+1)` → `3*(x+1)`, `(x+1)(x-1)` → `(x+1)*(x-1)`.
- Valida con una **máquina de estados** que alterna entre *"espero un operando"* y *"espero un operador"*. Detecta operadores seguidos, inicio o fin con operador, paréntesis vacíos o desbalanceados, funciones sin paréntesis y números pegados.

In [ ]:
%%writefile arbol_expresion/sintaxis.py
"""
sintaxis.py — Preprocesamiento y validación sintáctica.

preprocesar():
  - convierte el '-' unario en el operador 'neg' (y descarta el '+' unario)
  - inserta multiplicaciones implícitas: 2x, 3(x+1), (x+1)(x-1), 2sqrt(x)

validar_sintaxis(): máquina de estados que alterna entre
"espero un operando" y "espero un operador".
"""
from .config import TIPOS_OPERANDO
from .errores import ErrorExpresion, senalar, simbolo
from .lexico import Token


def preprocesar(tokens, expr):
    """Regresa (tokens_procesados, avisos)."""
    resultado, avisos = [], []
    for tok in tokens:
        prev = resultado[-1] if resultado else None

        # 1) Multiplicación implícita
        if prev is not None:
            termina_operando = prev.tipo in TIPOS_OPERANDO + ('PAR_DER',)
            inicia_operando = tok.tipo in TIPOS_OPERANDO + ('FUNC', 'PAR_IZQ')
            if termina_operando and inicia_operando:
                if prev.tipo == 'NUM' and tok.tipo == 'NUM':
                    raise ErrorExpresion(senalar(
                        expr, tok.pos, f"Falta un operador entre {prev.valor} y {tok.valor}."))
                resultado.append(Token('OP', '*', tok.pos))
                avisos.append(f"Multiplicación implícita insertada: {prev.valor} * {tok.valor}")
                prev = resultado[-1]

        # 2) Signo unario: al inicio, después de un operador o después de '('
        if tok.tipo == 'OP' and tok.valor in ('+', '-'):
            if prev is None or prev.tipo in ('OP', 'PAR_IZQ'):
                if tok.valor == '+':
                    continue
                tok = Token('OP', 'neg', tok.pos)

        resultado.append(tok)
    return resultado, avisos


def _error_esperando_operando(tok, anterior, siguiente, expr):
    """Valida un token cuando se esperaba un operando. Regresa True si ya llegó el operando."""
    if tok.tipo in TIPOS_OPERANDO:
        return True
    if tok.tipo == 'PAR_IZQ' or (tok.tipo == 'OP' and tok.valor == 'neg'):
        return False
    if tok.tipo == 'FUNC':
        if siguiente is None or siguiente.tipo != 'PAR_IZQ':
            raise ErrorExpresion(senalar(
                expr, tok.pos,
                f"La función '{tok.valor}' debe ir seguida de '('. Ejemplo: {tok.valor}(x)"))
        return False
    if tok.tipo == 'PAR_DER':
        msg = ("Paréntesis vacíos '()'." if anterior is not None and anterior.tipo == 'PAR_IZQ'
               else "Falta un operando antes de ')'.")
        raise ErrorExpresion(senalar(expr, tok.pos, msg))
    # Operador binario donde se esperaba un operando
    if anterior is None:
        msg = f"La expresión no puede iniciar con el operador '{tok.valor}'."
    elif anterior.tipo == 'OP':
        msg = f"Dos operadores seguidos: '{simbolo(anterior.valor)}' y '{tok.valor}'."
    else:
        msg = f"Falta un operando antes de '{tok.valor}'."
    raise ErrorExpresion(senalar(expr, tok.pos, msg))


def validar_sintaxis(tokens, expr):
    """Lanza ErrorExpresion si la secuencia de tokens no es una expresión válida."""
    if not tokens:
        raise ErrorExpresion("La expresión no contiene elementos válidos.")

    espera_operando = True
    abiertos = []                      # posiciones de '(' sin cerrar

    for i, tok in enumerate(tokens):
        anterior = tokens[i - 1] if i > 0 else None
        siguiente = tokens[i + 1] if i + 1 < len(tokens) else None

        if espera_operando:
            if tok.tipo == 'PAR_IZQ':
                abiertos.append(tok.pos)
            if _error_esperando_operando(tok, anterior, siguiente, expr):
                espera_operando = False
        elif tok.tipo == 'OP':
            espera_operando = True
        elif tok.tipo == 'PAR_DER':
            if not abiertos:
                raise ErrorExpresion(senalar(expr, tok.pos, "Paréntesis ')' sin su '(' correspondiente."))
            abiertos.pop()
        else:
            raise ErrorExpresion(senalar(expr, tok.pos, f"Falta un operador antes de '{tok.valor}'."))

    if espera_operando:
        ultimo = tokens[-1]
        raise ErrorExpresion(senalar(
            expr, ultimo.pos, f"La expresión está incompleta: termina en '{simbolo(ultimo.valor)}'."))
    if abiertos:
        n = len(abiertos)
        raise ErrorExpresion(senalar(
            expr, abiertos[-1], f"Falta{'n' if n > 1 else ''} {n} paréntesis de cierre ')'."))

## Paso 6 · Módulo `postfija.py` (Shunting-Yard)

Convierte la notación **infija** a **postfija** y guarda cada paso del algoritmo en una tabla (token, acción, pila, salida).

In [ ]:
%%writefile arbol_expresion/postfija.py
"""
postfija.py — Conversión infija -> postfija con el algoritmo Shunting-Yard (Dijkstra).

Reglas:
  - Operando           -> salida
  - Función o '('      -> pila
  - Operador binario   -> antes de apilarlo salen los de MAYOR precedencia
                          (o IGUAL si es asociativo por la izquierda)
  - Operador unario    -> se apila directamente (es prefijo)
  - ')'                -> desapilar hasta '('; si debajo hay función, también sale
  - Fin                -> vaciar la pila
"""
from .config import OPERADORES, TIPOS_OPERANDO
from .errores import ErrorExpresion


def _texto(lista):
    return ' '.join(t.valor for t in lista) if lista else '∅'


def a_postfija(tokens):
    """Regresa (postfija, pasos). 'pasos' es la tabla del algoritmo para mostrarla."""
    salida, pila, pasos = [], [], []

    def anotar(token, accion):
        pasos.append((token, accion, _texto(pila), _texto(salida)))

    for tok in tokens:
        if tok.tipo in TIPOS_OPERANDO:
            salida.append(tok)
            anotar(tok.valor, "Operando → salida")

        elif tok.tipo in ('FUNC', 'PAR_IZQ'):
            pila.append(tok)
            anotar(tok.valor, "Función → pila" if tok.tipo == 'FUNC' else "'(' → pila")

        elif tok.tipo == 'PAR_DER':
            while pila and pila[-1].tipo != 'PAR_IZQ':
                salida.append(pila.pop())
            if not pila:
                raise ErrorExpresion("Paréntesis desbalanceados.")
            pila.pop()                                    # descarta '('
            accion = "Desapilar hasta '('"
            if pila and pila[-1].tipo == 'FUNC':
                salida.append(pila.pop())
                accion += " y sacar la función"
            anotar(')', accion)

        elif tok.tipo == 'OP':
            prec, asoc, aridad = OPERADORES[tok.valor]
            sacados = 0
            if aridad == 2:
                while pila and pila[-1].tipo == 'OP':
                    prec_tope = OPERADORES[pila[-1].valor][0]
                    if prec_tope > prec or (prec_tope == prec and asoc == 'izq'):
                        salida.append(pila.pop())
                        sacados += 1
                    else:
                        break
            pila.append(tok)
            anotar(tok.valor, "Operador → pila" + (f" (antes salen {sacados})" if sacados else ""))

    while pila:
        tope = pila.pop()
        if tope.tipo == 'PAR_IZQ':
            raise ErrorExpresion("Paréntesis desbalanceados.")
        salida.append(tope)
    anotar('fin', "Vaciar la pila → salida")
    return salida, pasos


def imprimir_pasos(pasos):
    """Imprime la tabla de Shunting-Yard."""
    ancho = max(len("Pila"), *(len(p[2]) for p in pasos))
    encabezado = f"{'#':>3} │ {'Token':<6} │ {'Acción':<40} │ {'Pila':<{ancho}} │ Salida"
    print(encabezado)
    print('─' * (len(encabezado) + 20))
    for k, (tok, accion, pila, salida) in enumerate(pasos, 1):
        print(f"{k:>3} │ {tok:<6} │ {accion:<40} │ {pila:<{ancho}} │ {salida}")

## Paso 7 · Módulo `arbol.py` (estructura, construcción y recorridos)

| Recorrido | Orden de visita | Notación que produce |
|---|---|---|
| **Preorden** | raíz → izquierdo → derecho | prefija (polaca) |
| **Inorden** | izquierdo → raíz → derecho | infija (sin paréntesis) |
| **Postorden** | izquierdo → derecho → raíz | postfija (polaca inversa) |

Las funciones y el menos unario tienen un solo hijo, que se coloca como hijo **derecho**.

In [ ]:
%%writefile arbol_expresion/arbol.py
"""
arbol.py — Estructura del árbol de expresión, construcción y recorridos.

Construcción a partir de la postfija con una pila:
  - Operando         -> hoja, se apila
  - Operador binario -> se desapilan dos nodos (primero el DERECHO, luego el IZQUIERDO)
  - Función / neg    -> se desapila un nodo que queda como hijo DERECHO
Al final la pila debe tener exactamente un nodo: la raíz.

Recorridos:
  Preorden  = raíz, izquierdo, derecho  -> notación prefija (polaca)
  Inorden   = izquierdo, raíz, derecho  -> notación infija (sin paréntesis)
  Postorden = izquierdo, derecho, raíz  -> notación postfija (polaca inversa)
"""
from .config import TIPOS_OPERANDO
from .errores import ErrorExpresion


class Nodo:
    def __init__(self, valor, tipo, izq=None, der=None):
        self.valor = valor   # texto que se muestra en el nodo
        self.tipo = tipo     # NUM, VAR, CONST, OP (binario), OP_UN (neg), FUNC
        self.izq = izq
        self.der = der

    def es_hoja(self):
        return self.izq is None and self.der is None

    def __repr__(self):
        return f"Nodo({self.valor!r})"


def construir_arbol(postfija):
    """Construye el árbol a partir de una lista de tokens en postfija y regresa la raíz."""
    pila = []
    for tok in postfija:
        if tok.tipo in TIPOS_OPERANDO:
            pila.append(Nodo(tok.valor, tok.tipo))
        elif tok.tipo == 'FUNC' or tok.valor == 'neg':
            if not pila:
                raise ErrorExpresion(f"Falta el operando de '{tok.valor}'.")
            tipo = 'FUNC' if tok.tipo == 'FUNC' else 'OP_UN'
            pila.append(Nodo(tok.valor, tipo, der=pila.pop()))
        else:
            if len(pila) < 2:
                raise ErrorExpresion(f"Faltan operandos para '{tok.valor}'.")
            der, izq = pila.pop(), pila.pop()
            pila.append(Nodo(tok.valor, 'OP', izq, der))
    if len(pila) != 1:
        raise ErrorExpresion("Expresión mal formada: sobran o faltan operandos.")
    return pila[0]


# ---------- Recorridos ----------
def preorden(n):
    return [] if n is None else [n.valor] + preorden(n.izq) + preorden(n.der)


def inorden(n):
    return [] if n is None else inorden(n.izq) + [n.valor] + inorden(n.der)


def postorden(n):
    return [] if n is None else postorden(n.izq) + postorden(n.der) + [n.valor]


def infija_con_parentesis(n):
    """Reconstruye la expresión con todos los paréntesis (conserva el significado)."""
    if n.tipo in TIPOS_OPERANDO:
        return n.valor
    if n.tipo == 'FUNC':
        return f"{n.valor}({infija_con_parentesis(n.der)})"
    if n.tipo == 'OP_UN':
        return f"(-{infija_con_parentesis(n.der)})"
    return f"({infija_con_parentesis(n.izq)} {n.valor} {infija_con_parentesis(n.der)})"


# ---------- Estadísticas ----------
def altura(n):
    return 0 if n is None else 1 + max(altura(n.izq), altura(n.der))


def contar_nodos(n):
    return 0 if n is None else 1 + contar_nodos(n.izq) + contar_nodos(n.der)


def contar_hojas(n):
    if n is None:
        return 0
    return 1 if n.es_hoja() else contar_hojas(n.izq) + contar_hojas(n.der)


def variables(n):
    if n is None:
        return set()
    propia = {n.valor} if n.tipo == 'VAR' else set()
    return propia | variables(n.izq) | variables(n.der)

## Paso 8 · Módulo `evaluador.py` (evaluación con validación de dominio)

Recorre el árbol en postorden y detecta: raíz de negativo, logaritmo de un valor ≤ 0, tangente indefinida, división entre cero, potencias con resultado complejo y desbordamiento.

In [ ]:
%%writefile arbol_expresion/evaluador.py
"""
evaluador.py — Evaluación numérica del árbol (recorrido en postorden).

Validaciones de dominio:
  - raíz cuadrada de negativo
  - logaritmo de cero o negativo
  - tangente indefinida
  - división entre cero (incluye 0 elevado a negativo)
  - potencia de base negativa con exponente no entero (resultado complejo)
  - desbordamiento / resultados no finitos
"""
import math

from .config import FUNCIONES, CONSTANTES
from .errores import ErrorExpresion


def validar_valores(valores):
    """Revisa que 'valores' sea un dict de números finitos. Regresa una copia limpia."""
    if valores is None:
        return {}
    if not isinstance(valores, dict):
        raise ErrorExpresion("'valores' debe ser un diccionario, por ejemplo {'x': 2}.")
    limpios = {}
    for k, v in valores.items():
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            raise ErrorExpresion(f"El valor de '{k}' debe ser numérico (recibido: {v!r}).")
        if not math.isfinite(v):
            raise ErrorExpresion(f"El valor de '{k}' debe ser un número finito.")
        limpios[str(k).lower()] = float(v)
    return limpios


def _aplicar_funcion(nombre, a):
    if nombre == 'sqrt' and a < 0:
        raise ErrorExpresion(f"Error de dominio: sqrt({a:g}) — raíz cuadrada de un número negativo.")
    if nombre in ('log', 'ln') and a <= 0:
        raise ErrorExpresion(f"Error de dominio: {nombre}({a:g}) — el logaritmo requiere un valor mayor que 0.")
    if nombre == 'tan' and abs(math.cos(a)) < 1e-12:
        raise ErrorExpresion(f"Error de dominio: tan({a:g}) no está definida.")
    try:
        return FUNCIONES[nombre](a)
    except OverflowError:
        raise ErrorExpresion(f"Desbordamiento al calcular {nombre}({a:g}).")


def _aplicar_operador(op, a, b):
    if op == '/' and b == 0:
        raise ErrorExpresion(f"División entre cero: {a:g} / 0.")
    if op == '^':
        if a == 0 and b < 0:
            raise ErrorExpresion(f"División entre cero: 0^{b:g}.")
        if a < 0 and not float(b).is_integer():
            raise ErrorExpresion(f"Error de dominio: ({a:g})^{b:g} da un número complejo.")
    operaciones = {
        '+': lambda: a + b,
        '-': lambda: a - b,
        '*': lambda: a * b,
        '/': lambda: a / b,
        '^': lambda: a ** b,
    }
    if op not in operaciones:
        raise ErrorExpresion(f"Operador desconocido '{op}'.")
    try:
        r = operaciones[op]()
    except OverflowError:
        raise ErrorExpresion(f"Desbordamiento al calcular {a:g} {op} {b:g}.")
    if not math.isfinite(r):
        raise ErrorExpresion(f"El resultado de {a:g} {op} {b:g} no es un número finito.")
    return r


def evaluar(n, valores=None):
    """Evalúa el árbol con los valores de las variables. Lanza ErrorExpresion si falla."""
    valores = valores or {}
    if n.tipo == 'NUM':
        return float(n.valor)
    if n.tipo == 'CONST':
        return CONSTANTES[n.valor]
    if n.tipo == 'VAR':
        if n.valor not in valores:
            raise ErrorExpresion(f"Falta el valor de la variable '{n.valor}'.")
        return float(valores[n.valor])
    if n.tipo == 'OP_UN':
        return -evaluar(n.der, valores)
    if n.tipo == 'FUNC':
        return _aplicar_funcion(n.valor, evaluar(n.der, valores))
    return _aplicar_operador(n.valor, evaluar(n.izq, valores), evaluar(n.der, valores))

## Paso 9 · Módulo `graficos.py` (dibujo del árbol y de la función)

- **x** de cada nodo = su posición en el recorrido inorden (no se enciman); **y** = su profundidad.
- Los nodos unarios se centran sobre su hijo. Colores por tipo de nodo y aristas marcadas con **I** (izquierdo) o **D** (derecho).
- Si hay una sola variable, se grafica `f(variable)` omitiendo los puntos fuera del dominio.

In [ ]:
%%writefile arbol_expresion/graficos.py
"""
graficos.py — Dibujo del árbol y gráfica de la función con matplotlib.

Distribución del árbol:
  x = posición del nodo en el recorrido inorden (así nunca se enciman)
  y = profundidad del nodo
Los nodos unarios (funciones y neg) se centran encima de su único hijo.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from .config import COLORES
from .arbol import altura, infija_con_parentesis
from .errores import ErrorExpresion
from .evaluador import evaluar


def _calcular_posiciones(raiz):
    posiciones, nodos, contador = {}, [], [0]

    def asignar(n, prof):
        if n is None:
            return
        asignar(n.izq, prof + 1)
        posiciones[id(n)] = (contador[0], -prof)
        nodos.append(n)
        contador[0] += 1
        asignar(n.der, prof + 1)

    def centrar_unarios(n):
        if n is None:
            return
        centrar_unarios(n.izq)
        centrar_unarios(n.der)
        if n.tipo in ('FUNC', 'OP_UN'):
            posiciones[id(n)] = (posiciones[id(n.der)][0], posiciones[id(n)][1])

    asignar(raiz, 0)
    centrar_unarios(raiz)
    return posiciones, nodos


def dibujar_arbol(raiz, titulo="Árbol de expresión", guardar_como=None):
    """Dibuja el árbol. Si se da 'guardar_como', también lo guarda como imagen."""
    posiciones, nodos = _calcular_posiciones(raiz)
    h = altura(raiz)
    fig, ax = plt.subplots(figsize=(max(7, 1.0 * len(nodos)), max(4, 1.3 * h + 1)))

    # Aristas con la etiqueta I (izquierdo) o D (derecho)
    for n in nodos:
        x0, y0 = posiciones[id(n)]
        for hijo, lado in ((n.izq, 'I'), (n.der, 'D')):
            if hijo is not None:
                x1, y1 = posiciones[id(hijo)]
                ax.plot([x0, x1], [y0, y1], color='#555555', lw=1.6, zorder=1)
                ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.08, lado, fontsize=8,
                        color='#888888', ha='center', va='center', zorder=2)

    # Nodos coloreados por tipo
    for n in nodos:
        x, y = posiciones[id(n)]
        forma = 'circle,pad=0.45' if len(n.valor) <= 2 else 'round,pad=0.45'
        ax.text(x, y, n.valor, ha='center', va='center', fontsize=12, fontweight='bold',
                color='white', zorder=3,
                bbox=dict(boxstyle=forma, fc=COLORES[n.tipo][0], ec='black', lw=1.2))

    presentes = {n.tipo for n in nodos}
    leyenda = [Patch(fc=color, ec='black', label=etiqueta)
               for tipo, (color, etiqueta) in COLORES.items() if tipo in presentes]
    ax.legend(handles=leyenda, loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)

    ax.set_xlim(-1, len(nodos))
    ax.set_ylim(-(h - 1) - 0.6, 0.6)
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    if guardar_como:
        plt.savefig(guardar_como, dpi=200, bbox_inches='tight')
        print(f"💾 Árbol guardado como '{guardar_como}'")
    plt.show()


def graficar_funcion(raiz, var, rango=(-10, 10), puntos=1000):
    """Grafica f(var). Los puntos fuera del dominio se omiten."""
    try:
        a, b = rango
    except (TypeError, ValueError):
        print("⚠️ El rango debe ser una pareja (mínimo, máximo).")
        return
    if not all(isinstance(v, (int, float)) for v in (a, b)) or a >= b:
        print("⚠️ Rango inválido: debe ser (mínimo, máximo) con mínimo < máximo.")
        return

    xs = np.linspace(a, b, puntos)
    ys = []
    for x in xs:
        try:
            ys.append(evaluar(raiz, {var: x}))
        except ErrorExpresion:
            ys.append(np.nan)
    ys = np.array(ys, dtype=float)
    if np.all(np.isnan(ys)):
        print(f"⚠️ La función no está definida en ningún punto de [{a}, {b}].")
        return

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(xs, ys, lw=2, color='#4C72B0', label=f"f({var}) = {infija_con_parentesis(raiz)}")
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(alpha=0.3)
    ax.set_xlabel(var)
    ax.set_ylabel(f"f({var})")
    ax.set_title("Gráfica de la función (se omiten los puntos fuera del dominio)")
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    plt.show()

## Paso 10 · Módulo `analizador.py` (integración)

**Clase `ArbolExpresion`:** construye el árbol al crearse (si la expresión no es válida lanza `ErrorExpresion`) y ofrece los métodos
`preorden()`, `inorden()`, `postorden()`, `infija()`, `variables()`, `estadisticas()`, `evaluar()`, `mostrar_pasos()`, `dibujar()` y `graficar()`.

**Función `analizar()`:** imprime el reporte completo y nunca detiene el programa por un error de entrada.

In [ ]:
%%writefile arbol_expresion/analizador.py
"""
analizador.py — Punto de entrada: une todas las etapas.

  texto --(lexico)--> tokens --(sintaxis)--> tokens válidos
        --(postfija)--> postfija --(arbol)--> árbol
        --(evaluador / graficos)--> resultados

Dos formas de uso:
  1) Orientada a objetos:  a = ArbolExpresion("sqrt(2*(x^2)+5*x-3)")
                           a.preorden(); a.evaluar(x=2); a.dibujar()
  2) Reporte completo:     analizar("sqrt(2*(x^2)+5*x-3)", valores={'x': 2})
"""
from .errores import ErrorExpresion
from .lexico import normalizar, tokenizar
from .sintaxis import preprocesar, validar_sintaxis
from .postfija import a_postfija, imprimir_pasos
from . import arbol as _arbol
from .evaluador import evaluar as _evaluar, validar_valores
from .graficos import dibujar_arbol, graficar_funcion


class ArbolExpresion:
    """Árbol de expresión construido a partir de un texto. Lanza ErrorExpresion si el texto no es válido."""

    def __init__(self, expresion):
        self.original = expresion
        self.normalizada = normalizar(expresion)
        tokens = tokenizar(self.normalizada)
        self.tokens, self.avisos = preprocesar(tokens, self.normalizada)
        validar_sintaxis(self.tokens, self.normalizada)
        self.postfija, self.pasos = a_postfija(self.tokens)
        self.raiz = _arbol.construir_arbol(self.postfija)

    # ----- Recorridos -----
    def preorden(self):
        return _arbol.preorden(self.raiz)

    def inorden(self):
        return _arbol.inorden(self.raiz)

    def postorden(self):
        return _arbol.postorden(self.raiz)

    def infija(self):
        return _arbol.infija_con_parentesis(self.raiz)

    # ----- Información -----
    def variables(self):
        return sorted(_arbol.variables(self.raiz))

    def estadisticas(self):
        nodos = _arbol.contar_nodos(self.raiz)
        hojas = _arbol.contar_hojas(self.raiz)
        return {'nodos': nodos, 'hojas': hojas, 'internos': nodos - hojas,
                'altura': _arbol.altura(self.raiz), 'raiz': self.raiz.valor}

    def es_consistente(self):
        """El postorden del árbol debe coincidir con la postfija de Shunting-Yard."""
        return self.postorden() == [t.valor for t in self.postfija]

    # ----- Evaluación y gráficos -----
    def evaluar(self, valores=None, **kwargs):
        """Ejemplos: a.evaluar(x=2)   o   a.evaluar({'x': 2, 'y': 1})"""
        datos = dict(valores or {})
        datos.update(kwargs)
        return _evaluar(self.raiz, validar_valores(datos))

    def mostrar_pasos(self):
        imprimir_pasos(self.pasos)

    def dibujar(self, guardar_como=None):
        dibujar_arbol(self.raiz, titulo=f"Árbol de expresión: {self.normalizada}",
                      guardar_como=guardar_como)

    def graficar(self, rango=(-10, 10)):
        vars_ = self.variables()
        if len(vars_) != 1:
            print(f"⚠️ Solo se grafica si hay exactamente una variable (hay {len(vars_)}).")
            return
        graficar_funcion(self.raiz, vars_[0], rango=rango)

    def __repr__(self):
        return f"ArbolExpresion({self.normalizada!r})"


def construir(expresion):
    """Atajo: regresa un ArbolExpresion (lanza ErrorExpresion si no es válida)."""
    return ArbolExpresion(expresion)


def analizar(expr, valores=None, mostrar_pasos=True, graficar=True,
             graficar_f=True, rango=(-10, 10), guardar_como=None):
    """Imprime el reporte completo. Regresa el ArbolExpresion, o None si la expresión no es válida."""
    linea = '═' * 70
    print(f"{linea}\n📥 Expresión recibida: {expr}\n{linea}")

    try:
        a = ArbolExpresion(expr)
    except ErrorExpresion as err:
        print(f"❌ Expresión NO válida:\n   {err}")
        return None

    print(f"✅ Expresión válida\n   Normalizada: {a.normalizada}")
    for aviso in a.avisos:
        print(f"   ℹ️  {aviso}")

    print("\n🔤 TOKENS\n   " + '  '.join(f"[{t.valor}]" for t in a.tokens))

    if mostrar_pasos:
        print("\n🧮 CONVERSIÓN INFIJA → POSTFIJA (Shunting-Yard)\n")
        a.mostrar_pasos()

    print("\n🌳 RECORRIDOS DEL ÁRBOL")
    print(f"   Preorden  (raíz-izq-der, prefija) : {' '.join(a.preorden())}")
    print(f"   Inorden   (izq-raíz-der, infija)  : {' '.join(a.inorden())}")
    print(f"   Postorden (izq-der-raíz, postfija): {' '.join(a.postorden())}")
    print(f"   Infija con paréntesis             : {a.infija()}")
    print(f"\n🔎 Verificación: postorden == postfija de Shunting-Yard → "
          f"{'✔ coinciden' if a.es_consistente() else '✘ NO coinciden'}")

    e = a.estadisticas()
    vars_expr = a.variables()
    print("\n📊 ESTADÍSTICAS")
    print(f"   Nodos: {e['nodos']} | Hojas: {e['hojas']} | Nodos internos: {e['internos']} | "
          f"Altura: {e['altura']} | Raíz: '{e['raiz']}'")
    print(f"   Variables: {', '.join(vars_expr) if vars_expr else '(ninguna)'}")

    try:
        limpios = validar_valores(valores)
        sobrantes = set(limpios) - set(vars_expr)
        if sobrantes:
            print(f"   ⚠️ Se ignoran valores de variables que no aparecen: {', '.join(sorted(sobrantes))}")
        if not vars_expr or limpios:
            resultado = a.evaluar(limpios)
            asignacion = ', '.join(f"{k} = {limpios[k]:g}" for k in vars_expr)
            print(f"\n🧾 EVALUACIÓN{' con ' + asignacion if asignacion else ''}: {resultado:.10g}")
    except ErrorExpresion as err:
        print(f"\n⚠️ No se pudo evaluar: {err}")

    if graficar:
        a.dibujar(guardar_como=guardar_como)
    if graficar_f and len(vars_expr) == 1:
        a.graficar(rango=rango)
    return a

## Paso 11 · Módulo `interactivo.py`

In [ ]:
%%writefile arbol_expresion/interactivo.py
"""
interactivo.py — Modo interactivo por consola (funciona en Colab con input()).
"""
from .errores import ErrorExpresion
from .analizador import ArbolExpresion, analizar


def _pedir_valores(variables):
    """Pide el valor de cada variable. Regresa None si el usuario deja uno vacío."""
    valores = {}
    for v in variables:
        while True:
            texto = input(f"   Valor de {v} (Enter para no evaluar): ").strip().replace(',', '.')
            if texto == '':
                return None
            try:
                valores[v] = float(texto)
                break
            except ValueError:
                print("   ⚠️ Escribe un número válido, por ejemplo 2 o -3.5")
    return valores


def modo_interactivo(mostrar_pasos=True):
    print("Escribe una expresión (o 'salir' para terminar).")
    print("Ejemplos: sqrt(2*(x^2)+5*x-3) | (a+b)*(a-b) | 3x^2 - 2x + 1 | sin(x)/x\n")
    while True:
        expr = input("➤ Expresión: ").strip()
        if expr.lower() in ('salir', 'exit', 'q'):
            print("👋 Fin del modo interactivo.")
            break
        try:
            a = ArbolExpresion(expr)
        except ErrorExpresion as err:
            print(f"❌ {err}\n")
            continue
        valores = _pedir_valores(a.variables())
        analizar(expr, valores=valores, mostrar_pasos=mostrar_pasos)
        print()

## Paso 12 · Módulo `pruebas.py`

Prueba la validación sintáctica, la precedencia y asociatividad, la consistencia del árbol y los errores de dominio.

In [ ]:
%%writefile arbol_expresion/pruebas.py
"""
pruebas.py — Pruebas automáticas de validación, recorridos y evaluación.
"""
import math

from .errores import ErrorExpresion
from .analizador import ArbolExpresion

# (expresión, ¿debe ser válida?)
CASOS_SINTAXIS = [
    ("sqrt(2*(x^2)+5*x-3)", True),
    ("-x^2 + 3*(x - 1)/2", True),
    ("2^3^2", True),
    ("sin(pi/2) + ln(e)", True),
    ("(x+1)(x-1)", True),
    ("√(x)", True),
    ("", False),
    ("2*(x+3", False),
    ("2*x+3)", False),
    ("2*/x", False),
    ("*x+1", False),
    ("x+", False),
    ("()", False),
    ("sqrt x", False),
    ("raiz(x)", False),
    ("2 3 + x", False),
    ("3.4.5 + x", False),
    ("x & 2", False),
    ("y = 2x + 1", False),
]

# (expresión, valores, resultado esperado)
CASOS_EVALUACION = [
    ("sqrt(2*(x^2)+5*x-3)", {'x': 2}, math.sqrt(15)),
    ("2x^2 + 5x - 3", {'x': 2}, 15.0),
    ("2^3^2", {}, 512.0),          # asociatividad derecha
    ("-x^2", {'x': 3}, -9.0),      # el menos unario va después de la potencia
    ("10 - 4 - 3", {}, 3.0),       # asociatividad izquierda
    ("sin(pi/2) + ln(e)", {}, 2.0),
]

# (expresión, valores) que deben producir error de dominio
CASOS_DOMINIO = [
    ("sqrt(2*(x^2)+5*x-3)", {'x': 0}),
    ("1/(x-2)", {'x': 2}),
    ("ln(x)", {'x': -1}),
    ("(-8)^(1/3)", {}),
    ("exp(x)", {'x': 1000}),
]


def ejecutar_pruebas(detalle=True):
    total = aciertos = 0

    print("1) Validación sintáctica")
    for expr, esperado in CASOS_SINTAXIS:
        try:
            ArbolExpresion(expr)
            valida, msg = True, ""
        except ErrorExpresion as err:
            valida, msg = False, str(err).split('\n')[0]
        ok = valida == esperado
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            estado = "VÁLIDA  " if valida else "INVÁLIDA"
            print(f"   {'✔' if ok else '✘'} {estado} │ {expr!r:<24} │ {msg}")

    print("\n2) Evaluación y precedencia")
    for expr, vals, esperado in CASOS_EVALUACION:
        a = ArbolExpresion(expr)
        r = a.evaluar(vals)
        ok = math.isclose(r, esperado, rel_tol=1e-9) and a.es_consistente()
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con {vals} = {r:.6g} (esperado {esperado:.6g})")

    print("\n3) Errores de dominio")
    for expr, vals in CASOS_DOMINIO:
        try:
            ArbolExpresion(expr).evaluar(vals)
            ok, msg = False, "no se detectó el error"
        except ErrorExpresion as err:
            ok, msg = True, str(err)
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con {vals}: {msg}")

    print(f"\nResultado: {aciertos}/{total} pruebas correctas.")
    return aciertos, total

## Paso 13 · Módulo `__init__.py`

Define qué se puede importar directamente con `import arbol_expresion`.

In [ ]:
%%writefile arbol_expresion/__init__.py
"""
arbol_expresion — Paquete para construir, recorrer, evaluar y graficar árboles de expresión.

Módulos:
  config       Funciones, constantes, operadores y colores
  errores      Excepción ErrorExpresion y mensajes con señalamiento
  lexico       Normalización y tokenización
  sintaxis     Menos unario, multiplicación implícita y validación
  postfija     Algoritmo Shunting-Yard (infija -> postfija)
  arbol        Nodo, construcción, recorridos y estadísticas
  evaluador    Evaluación numérica con validación de dominio
  graficos     Dibujo del árbol y gráfica de la función
  analizador   Clase ArbolExpresion y función analizar()
  interactivo  Modo interactivo con input()
  pruebas      Pruebas automáticas
"""
from .errores import ErrorExpresion
from .analizador import ArbolExpresion, analizar, construir
from .interactivo import modo_interactivo
from .pruebas import ejecutar_pruebas

__all__ = ['ErrorExpresion', 'ArbolExpresion', 'analizar', 'construir',
           'modo_interactivo', 'ejecutar_pruebas']
__version__ = '2.0'

## Paso 14 · Importar el paquete

`autoreload` hace que, si modificas un módulo y vuelves a ejecutar su celda `%%writefile`, los cambios se carguen sin reiniciar el entorno.

In [ ]:
%load_ext autoreload
%autoreload 2

import arbol_expresion as ae
from arbol_expresion import ArbolExpresion, ErrorExpresion, analizar

print(f"✅ Paquete arbol_expresion v{ae.__version__} importado.")
print("   Archivos:", ', '.join(sorted(f for f in os.listdir('arbol_expresion') if f.endswith('.py'))))

## Paso 15 · Reporte completo con la ecuación de la imagen

$\sqrt{2x^2 + 5x - 3}$ evaluada en $x = 2$: $\sqrt{8 + 10 - 3} = \sqrt{15} \approx 3.873$

👉 **Para usar otra expresión, cambia `EXPRESION` y `VALORES` y vuelve a ejecutar esta celda.**

In [ ]:
EXPRESION = "sqrt(2*(x^2)+5*x-3)"
VALORES = {'x': 2}

arbol = analizar(EXPRESION, valores=VALORES, rango=(-8, 6), guardar_como="arbol_expresion.png")

### La segunda línea de la imagen, escrita "a mano"

`2x^2 + 5x - 3` usa multiplicación implícita; el programa la detecta y la convierte en `2*x^2 + 5*x - 3`.

In [ ]:
arbol2 = analizar("2x^2 + 5x - 3", valores={'x': 2}, mostrar_pasos=False, rango=(-5, 3))

## Paso 16 · Uso orientado a objetos con `ArbolExpresion`

Útil cuando solo necesitas una parte del resultado (por ejemplo, un recorrido o un valor).

In [ ]:
try:
    a = ArbolExpresion("sqrt(2*(x^2)+5*x-3)")
    print("Preorden :", ' '.join(a.preorden()))
    print("Inorden  :", ' '.join(a.inorden()))
    print("Postorden:", ' '.join(a.postorden()))
    print("Infija   :", a.infija())
    print("Variables:", a.variables())
    print("Estadísticas:", a.estadisticas())
    for x in [0.5, 1, 2, 5]:
        print(f"f({x}) = {a.evaluar(x=x):.6f}")
    a.evaluar(x=0)          # provoca un error de dominio a propósito
except ErrorExpresion as err:
    print("⚠️", err)

## Paso 17 · Pruebas automáticas de validaciones

In [ ]:
ae.ejecutar_pruebas()

## Paso 18 · Modo interactivo (opcional)

Escribe expresiones una por una. Si la expresión tiene variables, se piden sus valores (deja vacío para no evaluar). Escribe **`salir`** para terminar.

In [ ]:
ae.modo_interactivo()